# Failure modes — crosstab by category

How failure modes distribute across the three backend categories.

In [ ]:
from analysis import load_runs

# Pin matplotlib (Agg) + RNG seeds for reproducible, headless figures.
plt = load_runs.pin_style(seed=0)

# Load real runs if any exist, else the deterministic synthetic dataset.
records = load_runs.load_all()
backends = load_runs.backend_names(records)
STRATEGY = "zero_shot"  # the held-fixed strategy for the cross-backend tests
print(f"{len(records)} runs, backends={backends}")

In [ ]:
from collections import Counter, defaultdict

import numpy as np

from analysis import stats as st
from analysis.aggregate import aggregate_runs

summaries = aggregate_runs(records)  # cell-level view, for the run count below
table = defaultdict(Counter)  # category -> failure_mode -> count
totals = Counter()  # category -> total runs
for r in records:
    totals[str(r.backend.category)] += 1
    if r.failure_mode is not None:
        table[str(r.backend.category)][str(r.failure_mode)] += 1
categories = sorted(totals)
modes = sorted({m for c in table.values() for m in c})
print(f"{len(summaries)} cells; categories: {categories}")
print("failure modes:", modes)

In [ ]:
# Refusal rate per category with a small-n Wilson interval (the stats lens on
# the crosstab's REFUSAL row).
refusal_ci = {c: st.wilson_ci(table[c].get("refusal", 0), totals[c], backend=c) for c in categories}
for c, ci in refusal_ci.items():
    print(
        f"{c}: refusals {ci.successes}/{ci.n} = {ci.proportion:.2f} "
        f"[{ci.ci_low:.2f}, {ci.ci_high:.2f}]"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(categories))
cmap = plt.get_cmap("tab20")
for i, mode in enumerate(modes):
    heights = np.array([table[c].get(mode, 0) for c in categories], dtype=float)
    ax.bar(categories, heights, bottom=bottom, label=mode, color=cmap(i % 20))
    bottom += heights
ax.set_ylabel("count")
ax.set_title("Failure modes by backend category")
ax.legend(fontsize=8, ncol=2, bbox_to_anchor=(1.01, 1), loc="upper left")
load_runs.save_figure(fig, "06_failure_mode_crosstab", "failure_modes_by_category")
fig

In [ ]:
from IPython.display import Markdown

header = "| failure mode | " + " | ".join(categories) + " |"
sep = "|---" * (len(categories) + 1) + "|"
body = [f"| {m} | " + " | ".join(str(table[c].get(m, 0)) for c in categories) + " |" for m in modes]
refusals = "; ".join(f"{c}: {refusal_ci[c].proportion:.2f}" for c in categories)
Markdown(
    "### Summary — failure-mode crosstab\n\n"
    f"Refusal rate by category: {refusals}.\n\n" + "\n".join([header, sep, *body])
)